<a href="https://colab.research.google.com/github/eyobedb/Multimodal-papaya-disease-classification-Leveraging-Computer-vision-and-NLP/blob/main/Multimodal_ResNet50_GPT_2_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# =========================================
# Multimodal (ResNet50 + GPT-2) Classification in PyTorch
# =========================================

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# ------------------------------------------------------------
# 1. Device setup
# ------------------------------------------------------------

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# 2. Data Preparation
# ------------------------------------------------------------

In [ ]:

data_dir_train = "/content/dataset/train"
data_dir_val = "/content/dataset/val"
data_dir_test = "/content/dataset/test"

In [ ]:
img_size = 256
batch_size = 16

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder(data_dir_train, transform=transform)
val_dataset = datasets.ImageFolder(data_dir_val, transform=transform)
test_dataset = datasets.ImageFolder(data_dir_test, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

num_classes = len(train_dataset.classes)
print(f"Classes: {train_dataset.classes}")

# ------------------------------------------------------------
# 3. GPT-2 Text Encoder for Class Descriptions
# ------------------------------------------------------------

In [ ]:

print("\n🔹 Loading GPT-2...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
gpt2 = GPT2Model.from_pretrained("gpt2").to(device)
gpt2.eval()

class_descriptions = {
    "Black_spot_papaya": "Leaves covered with black spots indicating black spot infection.",
    "Powdery_mildew": "Papaya leaves showing powdery mildew infection typical of powdery disease.",
    "Ring_spot_papaya": "Ring spot disease on papaya leaves indicating ring spot infection.",
    "Healthy_papaya": "Healthy Papaya leaf with no visible signs of disease or damage."
}

# Compute GPT-2 embeddings for each class

In [ ]:

text_features = {}
with torch.no_grad():
    for cls, desc in class_descriptions.items():
        inputs = tokenizer(desc, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = gpt2(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1)
        text_features[cls] = emb.squeeze().cpu().numpy()

text_dim = emb.shape[-1]
print("Text feature dimension:", text_dim)

# Map text embeddings to each image in the dataset
def get_text_features_for_batch(labels):
    features = []
    for lbl in labels:
        cls_name = train_dataset.classes[lbl]
        features.append(text_features[cls_name])
    return torch.tensor(features, dtype=torch.float32)

# ------------------------------------------------------------
# 4. Define Multimodal Model
# ------------------------------------------------------------

In [ ]:

class MultimodalModel(nn.Module):
    def __init__(self, num_classes, text_dim=768):
        super(MultimodalModel, self).__init__()
        # Image Encoder: ResNet50
        self.resnet = models.resnet50(pretrained=True)
        for param in self.resnet.parameters():
            param.requires_grad = False  # Freeze ResNet
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 512)

        # Text Encoder
        self.text_fc = nn.Linear(text_dim, 512)

        # Fusion + Classifier
        self.fc1 = nn.Linear(1024, 256)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, img, txt):
        img_feat = self.resnet(img)
        txt_feat = torch.relu(self.text_fc(txt))
        combined = torch.cat((img_feat, txt_feat), dim=1)
        x = torch.relu(self.fc1(combined))
        x = self.dropout(x)
        return self.fc2(x)

model = MultimodalModel(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ------------------------------------------------------------
# 5. Training Loop
# ------------------------------------------------------------

In [ ]:

def train_model(model, train_loader, val_loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            imgs, labels = imgs.to(device), labels.to(device)
            text_feats = get_text_features_for_batch(labels).to(device)

            optimizer.zero_grad()
            outputs = model(imgs, text_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        acc = correct / total
        print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {acc*100:.2f}%")

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                text_feats = get_text_features_for_batch(labels).to(device)
                outputs = model(imgs, text_feats)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        val_acc = val_correct / val_total
        print(f"✅ Validation Accuracy: {val_acc*100:.2f}%\n")

train_model(model, train_loader, val_loader, epochs=5)

# ------------------------------------------------------------
# 6. Testing and Confusion Matrix
# ------------------------------------------------------------

In [ ]:
print("\n🔹 Evaluating on test set...")
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        text_feats = get_text_features_for_batch(labels).to(device)
        outputs = model(imgs, text_feats)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=train_dataset.classes,
            yticklabels=train_dataset.classes)
plt.title("Confusion Matrix - Multimodal ResNet50 + GPT-2")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


# Classification Report

In [ ]:
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=train_dataset.classes))